# L01 · Aurora 是什么——从正弦波（sine wave）到 Whisper，6 个月的路线图

**学习目标**
1. 理解 Aurora 为什么要从零实现，而不是调用现成 API
2. 看到完整的 11 模块路线图和每月通关标志
3. 用「我学 X 是为了能做 Y」的句式内化每个模块的意义
4. 实现 `check_imports` 和 `environment_report`——主动调试你的开发环境
5. 写下 3 个可在 6 个月后对答案的学习目标

## 1. 为什么从零写？

你用两行代码就能调用 Whisper：

```python
import openai
openai.audio.transcriptions.create(model="whisper-1", file=audio_file)
```

但这对面试没有帮助。面试官想问的是：

| 面试官的问题 | 能回答吗？ |
|---|---|
| Whisper 的 Mel filterbank 用多少个三角滤波器？为什么是 80？ | ❌ 用了 API 就不知道 |
| FFT 的时间复杂度为什么是 O(N log N)？蝶形网络（butterfly network）是什么？ | ❌ |
| CTC loss 如何处理未知对齐？forward-backward 算法是什么？ | ❌ |

Aurora 的方法：**自己写每一层**，然后用参考实现对答案。
6 个月后，上面三个问题都能在白板上推导。

In [ ]:
# 你能在 5 行 NumPy 里造出 Whisper 最底层所需的正弦波
import numpy as np
import matplotlib.pyplot as plt

sr, duration, freq = 16000, 0.02, 440  # 采样率, 时长(秒), A4 音
t = np.arange(round(duration * sr)) / sr
x = np.sin(2 * np.pi * freq * t)  # aurora.audio.sine 的核心就是这一行

plt.figure(figsize=(8, 2))
plt.plot(t * 1000, x)
plt.xlabel('时间 (ms)'); plt.ylabel('振幅')
plt.title(f'440 Hz 正弦波  ({len(t)} 个采样点)')
plt.tight_layout(); plt.show()
print(f'shape={x.shape}  dtype={x.dtype}  范围=[{x.min():.2f}, {x.max():.2f}]')

## 2. Aurora 的核心原则

> **当实现一个核心能力时，构建算法本身——不导入已经完成的黑盒。**

具体边界：

| 模块 | 允许 | 禁止 |
|---|---|---|
| audio_core | NumPy（数组容器 + 逐元素运算）| librosa、scipy.signal |
| deep_learning | PyTorch 自动微分（automatic differentiation，autograd） | `torch.nn` 当算法捷径 |
| ASR / Music | HuggingFace（推理框架）| Whisper API 当「我理解了 ASR」|
| LLM | HuggingFace transformers | ChatGPT API 当「我实现了 LLM」|

这条原则不是仪式——它是你面试时能讲清楚的那条边界。

## 3. 你将构建什么

旧目标（错误）：10000 个 commit、博客 20 篇、会用 10 个框架。
真正的目标：**少量、正确、能在面试中讲清楚的深度作品**——证据链，不是作品集数量。

6 个月，4 个可展示的系统：

```
① Audio Analysis Engine
   FFT + STFT + Mel filterbank + MFCC，全部手写，38 个测试全绿

② Keyword Spotting (KWS) Demo
   CNN 分类器，从波形到 yes/no，PyTorch 训练

③ ASR Transcript Engine
   CTC loss 手写，Whisper 架构理解，WER 评估

④ Podcast Intelligence (RAG + QA)
   TF-IDF 检索，RAG 流水线，Tool-calling Agent
```

每个系统都有：跑通的 demo、技术 blog 一篇、面试中能现场推导的核心算法。

## 4. 课程路线图（11 模块）

```
L01–L05  ① 开场          动机、声音表示、谱图直觉、三角函数、复数（complex number）
              ↓
L06–L08  ② Fourier 直觉  欧拉公式（Euler's formula） → Fourier 直觉 → 可视化
              ↓
L09–L21  ③ 线性代数（linear algebra）      向量、矩阵、特征值（eigenvalue）、SVD → DFT 矩阵视角
              ↓
L22–L26  ④ 微积分（calculus）        导数（derivative）、梯度（gradient）、链式法则（chain rule）、梯度下降（gradient descent）
              ↓
L27–L31  ⑤ 概率          分布（probability distribution）、交叉熵（cross entropy，CE）、Softmax
              ↓
L32–L53  ⑥ 音频 DSP      FFT ← 最重要一关：L37-L41 手写 FFT
              ↓
L54–L65  ⑦ 深度学习（deep learning，DL）      Autograd → MLP → PyTorch → KWS 训练
              ↓
L66–L75  ⑧ 语音识别（automatic speech recognition，ASR）      CTC → Whisper → WER 评估
              ↓
L76–L82  ⑨ 音乐          Music embedding → MusicGen / Suno 路径
              ↓
L83–L91  ⑩ 语言模型（language model，LLM）      Transformer → LoRA → RAG → Agent
              ↓
L92–L94  ⑪ 集成          异步流水线（asynchronous pipeline） → MLOps → 综合 Demo
```

**每一层的理解直接建立在上一层之上，不能跳。**
最关键一关：L37-L41（手写 FFT）——是整个 audio_core 的心脏。

## 5. 每月通关标志

路线图是静态地图；通关标志是「我确实到了这里」的证据。

| 月份 | 通关标志 | 交付物 |
|---|---|---|
| Month 1 | 能从空白文件重写 FFT / STFT / Mel / MFCC | Audio Analysis Engine |
| Month 2 | 训练出 KWS 模型，test accuracy > 90% | KWS Demo + W&B 训练曲线 |
| Month 3 | 能解释 CTC forward-backward，手写 WER 计算 | ASR Transcript Engine |
| Month 4 | 从头实现 Transformer attention，能 fine-tune | LM Core |
| Month 5 | RAG 流水线跑通，能解释 TF-IDF vs embedding | Podcast Intelligence |
| Month 6 | 综合 demo 跑通，面试题全部能白板推导 | 完整 Aurora v1 |

每月结束时，用这张表自检：通关标志是否真正达到，还是只是「运行过一遍代码」。

In [ ]:
# 验证你的课程目录结构是否完整
from pathlib import Path

# 自动定位 notebooks 根目录（兼容任意安装路径）
_here = Path.cwd()
nb_root = _here if _here.name != '0_foundation' else _here.parent
modules = sorted([d for d in nb_root.iterdir() if d.is_dir()])

total = 0
for mod in modules:
    nbs = sorted(mod.glob('*.ipynb'))
    total += len(nbs)
    if nbs:
        nums = [nb.stem.split('_')[0] for nb in nbs]
        print(f'  {mod.name:25s}  {nums[0]}–{nums[-1]}  ({len(nbs)} 课)')

print(f'\n总计：{total} 个 notebook')

## 6. ✏️ 练习：用自己的话写下每个模块的意义

路线图让你知道「去哪里」；这个练习让你知道「为什么去」。

**推理路线**：对每个模块，先想它的输入和输出是什么，
再想它对你的 Aurora 系统有什么具体贡献，
用 `"我学 X，是为了能在 Aurora 里做 Y"` 的句式写出来。

为什么要做这个练习？因为「看懂了路线图」和「能用自己的话解释每一步」
之间有一道沟——面试中考的是后者。

In [ ]:
course_purpose = {
    # ✏️ TODO: 把 None 替换成你自己的理解（字符串）
    # 示例：'1_complex_trig': '我学复数和三角函数，是为了理解 FFT 输出为什么是复数、模和相位分别代表什么'
    '1_complex_trig':   None,
    '2_linear_algebra': None,
    '3_calculus':       None,
    '4_probability':    None,
    '5_audio_dsp':      None,
    '6_deep_learning':  None,
    '7_asr':            None,
    '8_music':          None,
    '9_llm':            None,
    '10_integration':   None,
}

In [ ]:
unfilled = [k for k, v in course_purpose.items() if v is None]
assert not unfilled, f'还没填写：{unfilled}'
assert all(len(v) > 10 for v in course_purpose.values()), \
    '每条理解应超过 10 个字——写得更具体'

print('课程目的已记录：')
for mod, purpose in course_purpose.items():
    print(f'  {mod}: {purpose}')

## 7. 面试能力：每个模块能回答什么

| 面试问题 | 模块 |
|---|---|
| FFT 为什么是 O(N log N)？蝶形网络怎么画？ | L38-L39 |
| 采样率（sample rate，sr） 44100 Hz 是怎么来的？混叠（aliasing）是什么？ | L34 |
| Mel scale 为什么比线性频率更符合听觉？ | L46-L47 |
| CTC loss 怎么处理对齐？blank 符号的作用？ | L68-L69 |
| Attention 的时间复杂度？KV-Cache 怎么减少重复计算？ | L83, L85 |
| LoRA 为什么能减少参数量？rank 怎么选？ | L84 |
| RAG 的检索质量用什么指标评估？ | L89 |
| INT8 量化（quantization）误差如何计算？4bit 比 8bit 误差大多少？ | L87 |

Aurora 的目标：**让以上每一行都能在白板上推导，而不是「我用过这个库」**。

## 8. 如何使用这些 Notebook

| 操作 | 快捷键 |
|---|---|
| 运行当前 cell | `Shift + Enter` |
| 在下方插入 code cell | `B` |
| 切换为 Markdown cell | `M` |
| 切换为 Code cell | `Y` |

**约定**：
- `# ✏️ TODO` 标记是你需要填写的地方
- `assert` 语句会自动验证实现，全绿说明正确
- `# ← 改这里` 标记需要替换 `None`
- 参考实现在 `<details>` 折叠块里，建议先自己做再看

## 9. ✏️ 实现 `check_imports`——主动调试你的环境

想象你是一位医生，病人说"我感觉哪里不对"。
你不会说"那试试吃这个药"——你会先做一套检查，把问题精确到器官级别。

开发环境出问题时也一样。"import 失败"是症状，不是诊断。
`check_imports(names)` 就是那套检查工具：输入一组包名，输出哪个装了、哪个没装，
定位到包的粒度，而不是盯着 `ImportError` 发呆。

实现它：
- 用 `importlib.util.find_spec(name)` 检测每个包
- 返回 `None` = 未安装 → `False`；返回非 `None` → `True`
- 函数签名：`check_imports(names: list[str]) -> dict[str, bool]`

In [ ]:
import importlib.util

def check_imports(names):
    """检测一组包是否可导入。返回 {包名: bool} 字典。"""
    # ✏️ TODO: 对每个 name 用 importlib.util.find_spec 检测是否存在
    # find_spec 返回 None = 未安装；返回非 None = 可导入
    result = {}
    raise NotImplementedError("TODO: 使用 importlib.util.find_spec 检测每个包，返回 {name: bool} 字典")


In [ ]:
import importlib.util

try:
    status = check_imports(['numpy', 'matplotlib', 'aurora', 'not_a_real_package'])

    assert isinstance(status, dict),                  'check_imports 应返回字典'
    assert status.get('numpy') is True,               'numpy 应已安装'
    assert status.get('not_a_real_package') is False, '不存在的包应返回 False'

    print('check_imports 测试通过')
    print()

    required = ['numpy', 'matplotlib', 'aurora', 'ipykernel']
    result = check_imports(required)
    for name, ok in result.items():
        status_str = 'OK  ' if ok else 'FAIL'
        print(f'  {status_str}  {name}')

except NotImplementedError:
    print('⬜ check_imports 未实现，请填写函数体')
    print('   提示：importlib.util.find_spec(name) 返回 None 表示未安装')


In [ ]:
from aurora.audio.transforms import fft
import numpy as np
x = np.array([1.0, 0.0, -1.0, 0.0])
out = fft(x)
assert out.shape == (4,), f"Expected shape (4,), got {out.shape}"
assert np.isclose(out[0].real, 0.0, atol=1e-9), "DC component should be 0"
print("✅ aurora.audio.transforms.fft 验证通过")

### 当 `aurora` 返回 False 时，看哪三处

| 现象 | 优先检查 |
|---|---|
| `aurora: False` | 右上角内核名称是否为 `Python (AURORA)` |
| 内核已选正确但仍 False | 终端运行 `make install`，再重启内核 |
| `make install` 报错 | 检查是否在 AURORA 根目录，`pyproject.toml` 是否存在 |
| 安装成功但导入仍失败 | 运行 `import sys; print(sys.path)`，确认 `src/` 在路径里 |

**可编辑安装（editable install）的意义**：`make install` 执行了
`pip install -e .`，让 `import aurora` 直接连到 `src/aurora/`——
修改源码后无需重新安装，重启内核即可生效。

## ✏️ 热身推理：不运行代码，先想一想

在运行下面的 FFT 验证格之前，先回答这两个问题（写在下方注释里）：

**Q1**：信号 `[1, 0, -1, 0]` 只有 4 个采样点，代表的是什么频率？
- 采样率 sr = 1（归一化），信号长度 N = 4
- 提示：这个信号完成了几个完整的振荡周期？

**Q2**：FFT 输出 `out[0]` 是直流分量（DC，频率 = 0 Hz）。  
`[1, 0, -1, 0]` 的均值是多少？out[0].real 应该接近什么值？

> 把你的答案填在下面的代码格注释里，运行 FFT 后对答案。

In [ ]:
# ✏️ 填写你的预测答案（运行前写，运行后核对）
# Q1 答案：这个信号代表的频率是 ______（几倍的基频？）
# Q2 答案：out[0].real 预计约等于 ______

# 答案写完后运行下一格对答案

## 10. ✏️ 实现 `environment_report`——记录当前运行环境

开发中经常需要把「我的环境是什么」准确传达给别人（或者未来的自己）。
实现一个函数，返回包含环境信息的字典，作为你的「环境快照」。

需要包含的键：
- `python_executable`：当前 Python 解释器路径（`sys.executable`）
- `python_version`：版本字符串（`sys.version`）
- `aurora_available`：`aurora` 是否可导入（布尔值）
- `project_root`：包含 `pyproject.toml` 的目录路径（字符串）

In [ ]:
import sys
from pathlib import Path

def environment_report():
    """返回当前运行环境的快照字典。"""
    # ✏️ TODO: 填入四个键的值
    #
    # project_root 实现思路：
    #   从 Path.cwd() 开始，逐级向上检查父目录，
    #   找到第一个包含 pyproject.toml 的目录即为项目根目录。
    #   for folder in [Path.cwd(), *Path.cwd().parents]:
    #       if ...:
    #           root = ...; break
    root = None
    # ← 在这里实现 root 查找逻辑

    return {
        'python_executable': None,  # ← sys.executable
        'python_version':    None,  # ← sys.version
        'aurora_available':  None,  # ← 用 check_imports(['aurora'])['aurora']
        'project_root':      None,  # ← 用上面找到的 root（字符串）
    }


In [ ]:
import sys
from pathlib import Path

report = environment_report()

assert report['python_executable'] == sys.executable,  'python_executable 不正确'
assert report['python_version'] == sys.version,        'python_version 不正确'
assert isinstance(report['aurora_available'], bool),   'aurora_available 应为布尔值'
assert report['project_root'] is not None,             'project_root 未找到'
assert (Path(report['project_root']) / 'pyproject.toml').exists(), \
    f"project_root={report['project_root']} 下没有 pyproject.toml"

print('environment_report 通过')
print()
for k, v in report.items():
    val = str(v)[:80] + '...' if len(str(v)) > 80 else str(v)
    print(f'  {k}: {val}')


## 11. ✏️ 写下你的 3 个学习目标

刚才写了每个模块「对 Aurora 有什么贡献」；现在写它「对你有什么意义」。
6 个月后用来对答案。

In [ ]:
my_goals = {
    # ✏️ TODO: 把 None 换成你的目标（字符串）
    '技术目标': None,  # 例：'能从白板推导 FFT 的 O(N log N)'
    '项目目标': None,  # 例：'把 audio analysis engine 挂到 GitHub'
    '面试目标': None,  # 例：'能回答 FAANG 音频 / ML 系统设计面'
}

In [ ]:
assert all(v is not None for v in my_goals.values()), '每个目标都要填写'
assert all(isinstance(v, str) and len(v) > 5 for v in my_goals.values()), \
    '目标应为有意义的字符串（> 5 个字符）'
print('目标已记录：')
for k, v in my_goals.items():
    print(f'  {k}: {v}')

## 本课收束

本课建立了 6 个基础：

1. Aurora 原则：核心能力自己写，不导入黑盒
2. 证据链思维：少量深度作品 > 大量浅层堆砌
3. 路线图：11 模块的依赖顺序和不能跳的原因
4. 通关标志：每月有一个具体的「我真的到了」的证据
5. `check_imports` / `environment_report`：主动调试，而不是被动等待报错
6. 课程目的 + 个人目标：内化「为什么」，比「是什么」更重要

**下一课 L02**：声音的数字表示——采样率是什么，numpy 数组如何承载一段声音，
你将写出 `samples_count`、`make_time_axis`、`make_sine` 三个函数，
所有后续信号处理都从这里开始。

In [ ]:
# ✏️ 本课自评——运行前先填写，不要看答案
l01_review = {
    "aurora_no_api_principle": None,   # 我能用一句话解释为什么不用 API？ True/False
    "roadmap_dependency":      None,   # 我知道为什么不能跳过 L37-L41（FFT）？ True/False
    "check_imports_done":      None,   # check_imports 函数已实现并通过测试？ True/False
    "environment_report_done": None,   # environment_report 函数已实现并通过测试？ True/False
    "personal_goals_written":  None,   # 3 个个人目标已写下？ True/False
}

unfilled = [k for k, v in l01_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l01_review.items() if v is False]
if weak:
    print(f'⚠️  以下项需要复习再过一遍：{weak}')
else:
    print('✅ L01 全部通关！进入 L02：声音的数字表示')

---

→ **下一课**　[L02 · 声音的数字表示](L02_sound_digital.ipynb)

> 下节课将学习 **声音的数字表示**：采样定理、PCM 数组、第一个可听正弦波。